# Baseline Models on Washington RGB-D Object 51-Category

**Four fusion baselines using ResNet18 (full width):**
1. RGB-only (single stream, 3-channel input)
2. Depth-only (single stream, 1-channel input)
3. Early fusion (RGB+Depth concatenated = 4-channel input)
4. Late fusion (two backbones, features concatenated before classifier)

All models use identical training configuration (optimizer, scheduler, grad clipping, label smoothing, dropout, epochs).
Mean class accuracy (MCA) is the primary metric.

---

## Checklist Before Running:
- [ ] **Enable GPU:** Runtime > Change runtime type > A100 (preferred) or T4
- [ ] **Washington dataset on Drive:** `Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz` (produced by `notebooks/washington_preprocess.ipynb`)

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nShared drive contents:")
!ls -la /content/drive/Shareddrives/MSNN-Capstone/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "MSNN-Capstone"
GITHUB_REPO = "https://github.com/clingergab/MSNN-Capstone.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Copy Washington RGB-D Dataset to Local Disk

**Performance Note:** Local disk (`/dev/shm`) is ~10-20x faster than Drive!

**Dataset:** Washington RGB-D Object 51-category preprocessed (train + val + test splits, paired `.pt` tensors)

In [ ]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/washington_rgbd_256"  # Extracted location

print("=" * 60)
print("WASHINGTON RGB-D OBJECT DATASET SETUP (51 CATEGORIES)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")

    # Verify: count paired *_rgb.pt samples per split
    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"

    # Copy compressed file with progress
    !rsync -ah --info=progress2 "{DRIVE_DATASET_TAR}" {local_tar}

    # Extract to local disk
    print(f"\nExtracting dataset to local disk...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    # Remove tar file to save space
    !rm {local_tar}

    print(f"\nDataset extracted to local disk")

    # Verify extraction
    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

# Metadata files written by the preprocess notebook
for meta in ['class_names.txt', 'norm_stats.json']:
    meta_path = Path(LOCAL_DATASET_PATH) / meta
    status = 'OK' if meta_path.exists() else 'MISSING'
    print(f"   {meta}: {status}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

## 6. Setup Python Path & Imports

In [ ]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/MSNN-Capstone'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Importing core modules...")
from src.models.core.resnet import resnet18
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    get_washington_dataloaders,
    _load_class_names,
    _load_norm_stats,
    _discover_samples,
)
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler

print("All imports successful!")

In [ ]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 42
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 7. HPO Configuration

Ray Tune hyperparameter search settings. Each baseline runs as a separate experiment with 50 trials.

In [ ]:
import os
import subprocess
import time as _time
from pathlib import Path

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

# ======================== HPO SETTINGS ========================
DRIVE_STORAGE_PATH = "/content/drive/Shareddrives/MSNN-Capstone/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_PREFIX = "washington_baseline_hpo"

NUM_SAMPLES = 50        # Trials per baseline

# ======================== FIXED MODEL SETTINGS ========================
FIXED_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 51,
    'width_multiplier': 0.75,
    'epochs': 50,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'batch_size': 64,
    'num_workers': 1,
    'crop_size': 224,
}

# ======================== SEARCH SPACE ========================
BASE_SEARCH_SPACE = {
    "lr": tune.loguniform(5e-5, 5e-4),
    "wd": tune.loguniform(3e-5, 6e-4),
    "eta_min": tune.loguniform(0.001, 0.015),
    "dropout_p": tune.quniform(0.25, 0.50, 0.01),
    "label_smoothing": tune.quniform(0.05, 0.15, 0.01),
    "grad_clip_norm": tune.quniform(0.8, 1.5, 0.05),
    "rgb_aug_prob": tune.quniform(0.8, 1.2, 0.01),
    "rgb_aug_mag": tune.quniform(0.8, 1.2, 0.01),
    "depth_aug_prob": tune.quniform(0.8, 1.2, 0.01),
    "depth_aug_mag": tune.quniform(0.8, 1.2, 0.01),
}

Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

print("HPO Configuration:")
print(f"  Storage: {DRIVE_STORAGE_PATH}")
print(f"  Trials per baseline: {NUM_SAMPLES}")
print(f"  Fixed: ResNet18 {FIXED_CONFIG['width_multiplier']}x, {FIXED_CONFIG['epochs']} epochs")
print(f"  Search space: {len(BASE_SEARCH_SPACE)} parameters")

## 8. Load Dataset

In [ ]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'val', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        n = len(list(split_dir.glob('*/*_rgb.pt')))
        n_classes = len([d for d in split_dir.iterdir() if d.is_dir()])
        print(f"    {split}/ - {n} paired samples across {n_classes} class folders")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    class_names = _load_class_names(LOCAL_DATASET_PATH)
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

In [ ]:
print("=" * 60)
print("LOADING WASHINGTON RGB-D OBJECT DATASET (TRAIN + VAL + TEST)")
print("=" * 60)

print(f"\nLoading dataset from: {LOCAL_DATASET_PATH}")

train_loader, val_loader, test_loader, num_classes = get_washington_dataloaders(
    data_root=LOCAL_DATASET_PATH,
    batch_size=FIXED_CONFIG['batch_size'],
    num_workers=FIXED_CONFIG['num_workers'],
    crop_size=FIXED_CONFIG['crop_size'],
    seed=SEED,
    normalize=True,
    balanced_sampling=True,
)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

assert num_classes == FIXED_CONFIG['num_classes'], (
    f"num_classes mismatch: dataset has {num_classes}, "
    f"FIXED_CONFIG expects {FIXED_CONFIG['num_classes']}."
)

print(f"\nTrain samples: {len(train_loader.dataset):,}")
print(f"Val samples:   {len(val_loader.dataset):,}")
print(f"Test samples:  {len(test_loader.dataset):,}")
print(f"Batch size:    {FIXED_CONFIG['batch_size']}")
print(f"Norm stats loaded: {list(norm_stats.keys())}")

## 8b. Baseline HPO: ResNet18 (0.75x width)

**Four baselines** tuned independently with Ray Tune (50 trials each, ASHA early stopping):
1. **RGB-only:** Standard 3-channel input
2. **Depth-only:** Single-channel depth input
3. **Early Fusion:** 4-channel concatenated RGB+Depth
4. **Late Fusion:** Dual backbone with feature concatenation

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.models.core.resnet import resnet18
import ray
from src.training.augmentation_config import AugmentationConfig
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    _load_class_names,
    _discover_samples,
)

# --- Dataset wrappers ---
class RGBOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (rgb, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return rgb, label

class DepthOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (depth, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return depth, label

class EarlyFusionDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (cat(rgb, depth), label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return torch.cat([rgb, depth], dim=0), label


class LateFusionResNet(nn.Module):
    def __init__(self, num_classes, width_multiplier, dropout_p):
        super().__init__()
        self.rgb_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone.conv1 = nn.Conv2d(
            1, self.depth_backbone.conv1.out_channels,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        nn.init.kaiming_normal_(self.depth_backbone.conv1.weight, mode='fan_out', nonlinearity='relu')

        feat_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()
        self.depth_backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(feat_dim * 2, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)
        fused = torch.cat([rgb_feat, depth_feat], dim=1)
        return self.classifier(fused)


def _build_model(fusion_type, num_classes, width_multiplier, dropout_p):
    """Build baseline model for given fusion type."""
    if fusion_type == 'rgb':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'depth':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        old_conv1 = model.conv1
        model.conv1 = nn.Conv2d(1, old_conv1.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'early_fusion':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        old_conv1 = model.conv1
        model.conv1 = nn.Conv2d(4, old_conv1.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'late_fusion':
        return LateFusionResNet(num_classes, width_multiplier, dropout_p)

    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")


def _wrap_dataset(dataset, fusion_type):
    """Wrap base dataset for the given fusion type."""
    if fusion_type == 'rgb':
        return RGBOnlyDataset(dataset)
    elif fusion_type == 'depth':
        return DepthOnlyDataset(dataset)
    elif fusion_type == 'early_fusion':
        return EarlyFusionDataset(dataset)
    elif fusion_type == 'late_fusion':
        return dataset  # returns (rgb, depth, label) directly
    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")


def train_baseline_tune(config, fusion_type=None, data_root=None, norm_stats=None, seed=42,
                        class_names=None, train_samples=None, val_samples=None):
    """Ray Tune trainable for baseline HPO."""
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)
    device = 'cuda'
    num_classes = FIXED_CONFIG['num_classes']
    epochs = FIXED_CONFIG['epochs']
    crop_size = FIXED_CONFIG['crop_size']

    # Per-trial augmentation
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob", 1.0),
        rgb_aug_mag=config.get("rgb_aug_mag", 1.0),
        depth_aug_prob=config.get("depth_aug_prob", 1.0),
        depth_aug_mag=config.get("depth_aug_mag", 1.0),
    )

    # Datasets: train with aug, val without aug. Train/val come from on-disk splits.
    train_dataset = WashingtonRGBDDataset(
        data_root=data_root, split='train',
        samples=train_samples, class_names=class_names, norm_stats=norm_stats,
        crop_size=crop_size, normalize=True,
        **aug_config.to_dict(),
    )
    val_dataset = WashingtonRGBDDataset(
        data_root=data_root, split='val',
        samples=val_samples, class_names=class_names, norm_stats=norm_stats,
        crop_size=crop_size, normalize=True,
    )

    train_wrapped = _wrap_dataset(train_dataset, fusion_type)
    val_wrapped = _wrap_dataset(val_dataset, fusion_type)

    # Weighted sampler for class-balanced training
    sample_weights = train_dataset.get_sample_weights()
    train_sampler = WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights),
        replacement=True, generator=g)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    train_dl = DataLoader(train_wrapped, batch_size=FIXED_CONFIG['batch_size'],
                          shuffle=False, sampler=train_sampler,
                          num_workers=1, prefetch_factor=2,
                          persistent_workers=True, pin_memory=True,
                          worker_init_fn=worker_init_fn)
    val_dl = DataLoader(val_wrapped, batch_size=FIXED_CONFIG['batch_size'],
                        shuffle=False, num_workers=1, prefetch_factor=2,
                        persistent_workers=False, pin_memory=True,
                        worker_init_fn=worker_init_fn)

    is_late_fusion = (fusion_type == 'late_fusion')

    # Build model
    model = _build_model(fusion_type, num_classes,
                         FIXED_CONFIG['width_multiplier'], config['dropout_p'])
    model.to(device)

    # Optimizer & scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                  weight_decay=config['wd'])
    warmup = LinearLR(optimizer, start_factor=FIXED_CONFIG['warmup_start_factor'],
                      end_factor=1.0, total_iters=FIXED_CONFIG['warmup_epochs'])
    cosine = CosineAnnealingLR(optimizer,
                               T_max=FIXED_CONFIG['epochs'] - FIXED_CONFIG['warmup_epochs'],
                               eta_min=config['eta_min'])
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[FIXED_CONFIG['warmup_epochs']])

    criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
    scaler = torch.amp.GradScaler('cuda', enabled=True)

    best_val_mca = 0.0
    best_val_acc = 0.0
    best_val_loss = float('inf')

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        if is_late_fusion:
            for rgb, depth, labels in train_dl:
                rgb = rgb.to(device, non_blocking=True)
                depth = depth.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    logits = model(rgb, depth)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)
        else:
            for inputs, labels in train_dl:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    logits = model(inputs)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)

        scheduler.step()
        train_loss = total_loss / total
        train_acc = correct / total

        # --- Validate ---
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        v_preds, v_labels_all = [], []

        with torch.no_grad():
            if is_late_fusion:
                for rgb, depth, labels in val_dl:
                    rgb = rgb.to(device, non_blocking=True)
                    depth = depth.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda'):
                        logits = model(rgb, depth)
                        loss = criterion(logits, labels)
                    v_loss += loss.item() * labels.size(0)
                    preds = logits.argmax(1)
                    v_correct += preds.eq(labels).sum().item()
                    v_total += labels.size(0)
                    v_preds.append(preds.cpu())
                    v_labels_all.append(labels.cpu())
            else:
                for inputs, labels in val_dl:
                    inputs = inputs.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda'):
                        logits = model(inputs)
                        loss = criterion(logits, labels)
                    v_loss += loss.item() * labels.size(0)
                    preds = logits.argmax(1)
                    v_correct += preds.eq(labels).sum().item()
                    v_total += labels.size(0)
                    v_preds.append(preds.cpu())
                    v_labels_all.append(labels.cpu())

        val_loss = v_loss / v_total
        val_acc = v_correct / v_total
        v_preds = torch.cat(v_preds)
        v_labels_all = torch.cat(v_labels_all)
        per_class = [(v_preds[v_labels_all == c] == c).float().mean().item()
                     for c in range(num_classes) if (v_labels_all == c).sum() > 0]
        val_mca = sum(per_class) / len(per_class)

        # Track best
        if val_mca > best_val_mca:
            best_val_mca = val_mca
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Report to Ray Tune
        tune.report({
            "val_accuracy": val_acc,
            "val_loss": val_loss,
            "val_mca": val_mca,
            "best_val_mca": best_val_mca,
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
        })


# --- Drive sync callback ---
class DriveSyncCallback(TuneCallback):
    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a", self._local_path + "/", self._drive_path + "/"],
                capture_output=True, text=True, timeout=120)
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced ({reason})")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    def __init__(self, metric="best_val_mca", mode="max", every_n_results=10):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()
        if self._result_count % self._every_n == 0 and self._best_config is not None:
            print(f"\n{'---'*20}")
            print(f"  Best {self._metric}: {self._best_value*100:.2f}% ({self._result_count} results)")
            print(f"{'---'*20}")


def run_baseline_hpo(fusion_type, data_root, norm_stats, class_names,
                     train_samples, val_samples, search_space):
    """Run Ray Tune HPO for a single baseline fusion type."""
    experiment_name = f"{EXPERIMENT_PREFIX}_{fusion_type}"
    experiment_path = os.path.join(DRIVE_STORAGE_PATH, experiment_name)
    local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, experiment_name)

    # Check for existing experiment to resume
    resume_existing = os.path.exists(experiment_path)
    if resume_existing:
        _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
        if len(_exp_files) == 0:
            resume_existing = False

    if resume_existing:
        print(f"  Previous experiment found - restoring from Drive...")
        os.makedirs(local_experiment_path, exist_ok=True)
        subprocess.run(["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
                       capture_output=True, text=True)

    ray.shutdown()
    ray.init(ignore_reinit_error=True, runtime_env={
        "env_vars": {
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    })

    trainable = tune.with_resources(
        tune.with_parameters(
            train_baseline_tune,
            fusion_type=fusion_type,
            data_root=data_root,
            norm_stats=norm_stats,
            seed=SEED,
            class_names=class_names,
            train_samples=train_samples,
            val_samples=val_samples,
        ),
        resources={"cpu": 3, "gpu": 1.0 / 14},
    )

    drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, experiment_name)

    if resume_existing:
        tuner = tune.Tuner.restore(
            path=local_experiment_path,
            trainable=trainable,
            resume_unfinished=True,
            resume_errored=True,
        )
    else:
        reporter = CLIReporter(
            parameter_columns=["lr", "wd", "eta_min", "dropout_p", "label_smoothing", "grad_clip_norm", "rgb_aug_prob", "rgb_aug_mag", "depth_aug_prob", "depth_aug_mag"],
            metric_columns={
                "training_iteration": "iter",
                "best_val_mca": "best_val_mca",
                "best_val_acc": "best_val_acc",
            },
            max_report_frequency=30,
            print_intermediate_tables=True,
        )

        optuna_search = OptunaSearch(
            metric="best_val_mca", mode="max",
            sampler=TPESampler(n_startup_trials=10),
        )

        asha_scheduler = ASHAScheduler(
            time_attr="training_iteration",
            metric="best_val_mca", mode="max",
            max_t=FIXED_CONFIG['epochs'],
            grace_period=15,
            reduction_factor=2,
        )

        tuner = tune.Tuner(
            trainable,
            param_space=search_space,
            tune_config=tune.TuneConfig(
                scheduler=asha_scheduler,
                search_alg=optuna_search,
                num_samples=NUM_SAMPLES,
                max_concurrent_trials=14,
            ),
            run_config=ray.tune.RunConfig(
                storage_path=LOCAL_STORAGE_PATH,
                name=experiment_name,
                progress_reporter=reporter,
                verbose=1,
                callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=10)],
            ),
        )

    results = tuner.fit()

    best = results.get_best_result("best_val_mca", "max")
    print(f"\nBest {fusion_type} config: {best.config}")
    print(f"Best {fusion_type} val MCA: {best.metrics['best_val_mca']:.4f}")
    print(f"Best {fusion_type} val acc: {best.metrics['best_val_acc']:.4f}")

    return results


# Pre-discover samples once (avoids filesystem walk per trial)
class_names = _load_class_names(LOCAL_DATASET_PATH)
train_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'train'), class_names)
val_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'val'), class_names)
print(f"Pre-discovered samples: train={len(train_samples)}, val={len(val_samples)}")
print("Trainable function and helpers defined.")
print("Fusion types: rgb, depth, early_fusion, late_fusion")

In [ ]:
# --- RGB-Only HPO ---
print('=' * 60)
print(f"HPO: ResNet18 {FIXED_CONFIG['width_multiplier']}x width - RGB Only ({NUM_SAMPLES} trials)")
print('=' * 60)

# RGB-only: no depth augmentation search
rgb_search_space = {k: v for k, v in BASE_SEARCH_SPACE.items()
                    if not k.startswith('depth_aug')}
rgb_search_space['depth_aug_prob'] = 1.0
rgb_search_space['depth_aug_mag'] = 1.0

rgb_results = run_baseline_hpo('rgb', LOCAL_DATASET_PATH, norm_stats, class_names,
                                train_samples, val_samples, rgb_search_space)

# Top-10 trials by best_val_mca (one row per trial)
df = rgb_results.get_dataframe()
metric_cols = [c for c in ['best_val_mca', 'best_val_acc', 'best_val_loss', 'training_iteration']
               if c in df.columns]
config_cols = sorted(c for c in df.columns if c.startswith('config/'))
display_cols = metric_cols + config_cols
top_10 = df.sort_values('best_val_mca', ascending=False).head(10)[display_cols]
print(f"\nTop 10 RGB-Only trials by best_val_mca (out of {len(df)}):")
print(top_10.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

In [ ]:
# RGB-Only best trial summary
rgb_best = rgb_results.get_best_result("best_val_mca", "max")
print("\nRGB-Only Best Trial:")
for k, v in rgb_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {rgb_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {rgb_best.metrics['best_val_acc']:.4f}")

In [ ]:
# --- Depth-Only HPO ---
print('=' * 60)
print(f"HPO: ResNet18 {FIXED_CONFIG['width_multiplier']}x width - Depth Only ({NUM_SAMPLES} trials)")
print('=' * 60)

# Depth-only: no RGB augmentation search
depth_search_space = {k: v for k, v in BASE_SEARCH_SPACE.items()
                      if not k.startswith('rgb_aug')}
depth_search_space['rgb_aug_prob'] = 1.0
depth_search_space['rgb_aug_mag'] = 1.0

depth_results = run_baseline_hpo('depth', LOCAL_DATASET_PATH, norm_stats, class_names,
                                  train_samples, val_samples, depth_search_space)

# Top-10 trials by best_val_mca (one row per trial)
df = depth_results.get_dataframe()
metric_cols = [c for c in ['best_val_mca', 'best_val_acc', 'best_val_loss', 'training_iteration']
               if c in df.columns]
config_cols = sorted(c for c in df.columns if c.startswith('config/'))
display_cols = metric_cols + config_cols
top_10 = df.sort_values('best_val_mca', ascending=False).head(10)[display_cols]
print(f"\nTop 10 Depth-Only trials by best_val_mca (out of {len(df)}):")
print(top_10.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

In [ ]:
# Depth-Only best trial summary
depth_best = depth_results.get_best_result("best_val_mca", "max")
print("\nDepth-Only Best Trial:")
for k, v in depth_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {depth_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {depth_best.metrics['best_val_acc']:.4f}")

In [ ]:
# --- Early Fusion HPO ---
print('=' * 60)
print(f"HPO: ResNet18 {FIXED_CONFIG['width_multiplier']}x width - Early Fusion ({NUM_SAMPLES} trials)")
print('=' * 60)

# Early fusion uses all augmentation params
ef_results = run_baseline_hpo('early_fusion', LOCAL_DATASET_PATH, norm_stats, class_names,
                               train_samples, val_samples, BASE_SEARCH_SPACE.copy())

# Top-10 trials by best_val_mca (one row per trial)
df = ef_results.get_dataframe()
metric_cols = [c for c in ['best_val_mca', 'best_val_acc', 'best_val_loss', 'training_iteration']
               if c in df.columns]
config_cols = sorted(c for c in df.columns if c.startswith('config/'))
display_cols = metric_cols + config_cols
top_10 = df.sort_values('best_val_mca', ascending=False).head(10)[display_cols]
print(f"\nTop 10 Early Fusion trials by best_val_mca (out of {len(df)}):")
print(top_10.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

In [ ]:
# Early Fusion best trial summary
ef_best = ef_results.get_best_result("best_val_mca", "max")
print("\nEarly Fusion Best Trial:")
for k, v in ef_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {ef_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {ef_best.metrics['best_val_acc']:.4f}")


In [ ]:
# --- Late Fusion HPO ---
print('=' * 60)
print(f"HPO: ResNet18 {FIXED_CONFIG['width_multiplier']}x width - Late Fusion ({NUM_SAMPLES} trials)")
print('=' * 60)

# Late fusion uses all augmentation params
lf_results = run_baseline_hpo('late_fusion', LOCAL_DATASET_PATH, norm_stats, class_names,
                               train_samples, val_samples, BASE_SEARCH_SPACE.copy())

# Top-10 trials by best_val_mca (one row per trial)
df = lf_results.get_dataframe()
metric_cols = [c for c in ['best_val_mca', 'best_val_acc', 'best_val_loss', 'training_iteration']
               if c in df.columns]
config_cols = sorted(c for c in df.columns if c.startswith('config/'))
display_cols = metric_cols + config_cols
top_10 = df.sort_values('best_val_mca', ascending=False).head(10)[display_cols]
print(f"\nTop 10 Late Fusion trials by best_val_mca (out of {len(df)}):")
print(top_10.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

In [ ]:
# Late Fusion best trial summary
lf_best = lf_results.get_best_result("best_val_mca", "max")
print("\nLate Fusion Best Trial:")
for k, v in lf_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {lf_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {lf_best.metrics['best_val_acc']:.4f}")

## 9. HPO Results Comparison

In [ ]:
import matplotlib.pyplot as plt

# Collect best results
all_results = {
    'RGB-Only': rgb_results,
    'Depth-Only': depth_results,
    'Early Fusion': ef_results,
    'Late Fusion': lf_results,
}

# --- Best trial MCA bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(all_results.keys())
best_mcas = []
best_accs = []

for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    best_mcas.append(best.metrics['best_val_mca'] * 100)
    best_accs.append(best.metrics['best_val_acc'] * 100)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

axes[0].bar(names, best_mcas, color=colors)
axes[0].set_ylabel('Val MCA (%)')
axes[0].set_title('Best Val Mean Class Accuracy')
for i, v in enumerate(best_mcas):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

axes[1].bar(names, best_accs, color=colors)
axes[1].set_ylabel('Val Accuracy (%)')
axes[1].set_title('Best Val Accuracy')
for i, v in enumerate(best_accs):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Baseline HPO Summary

In [ ]:
# --- Summary table ---
print('=' * 80)
print(f"BASELINE HPO RESULTS - ResNet18 ({FIXED_CONFIG['width_multiplier']}x width) on Washington RGB-D Object 51-Category")
print('=' * 80)

print(f"{'Model':<18} {'Val MCA':>10} {'Val Acc':>10} {'Best LR':>12} {'Best WD':>12} {'Dropout':>10}")
print('-' * 80)

for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    m = best.metrics
    c = best.config
    print(f"{name:<18} {m['best_val_mca']*100:>9.2f}% {m['best_val_acc']*100:>9.2f}% "
          f"{c['lr']:>12.2e} {c['wd']:>12.2e} {c['dropout_p']:>9.2f}")

print('=' * 80)
print()

# Print full best configs
for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    print(f"\n--- {name} best config ---")
    for k, v in sorted(best.config.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.6f}" if abs(v) >= 0.01 else f"  {k}: {v:.2e}")
        else:
            print(f"  {k}: {v}")